[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/langchain-ai/langchain-academy/blob/main/module-3/breakpoints.ipynb) [![Open in LangChain Academy](https://cdn.prod.website-files.com/65b8cd72835ceeacd4449a53/66e9eba12c7b7688aa3dbb5e_LCA-badge-green.svg)](https://academy.langchain.com/courses/take/intro-to-langgraph/lessons/58239469-lesson-2-breakpoints)

# Breakpoints（断点）

## 回顾

在"人机协同"（`human-in-the-loop`）场景中，我们通常希望在图运行过程中查看其输出。

我们通过流式传输（streaming）为此奠定了基础。

## 目标

现在，让我们讨论"人机协同"的动机：

(1) `Approval`（审批）—— 我们可以中断 Agent，将状态展示给用户，并允许用户批准某个操作

(2) `Debugging`（调试）—— 我们可以回退图的执行以重现或避免问题

(3) `Editing`（编辑）—— 你可以修改状态

LangGraph 提供了多种获取或更新 Agent 状态的方法，以支持各种"人机协同"工作流。

首先，我们将介绍 [breakpoints](https://docs.langchain.com/oss/python/langgraph/interrupts#debugging-with-interrupts)（断点），它提供了一种在特定步骤停止图执行的简单方式。

我们将展示这如何支持用户 `approval`（审批）。

In [ ]:
%%capture --no-stderr
%pip install --quiet -U langgraph langchain_deepseek langgraph_sdk langgraph-prebuilt

In [ ]:
import os, getpass
from dotenv import load_dotenv

load_dotenv()

def _set_env(var: str):
    if not os.environ.get(var):
        os.environ[var] = getpass.getpass(f"{var}: ")

_set_env("DEEPSEEK_API_KEY")

## 用于人工审批的断点

让我们重新回顾模块 1 中使用的简单 Agent。

假设我们关心工具调用：我们希望在使用任何工具之前获得审批。

我们只需要在编译图时设置 `interrupt_before=["tools"]`，其中 `tools` 是我们的工具节点。

这意味着执行将在 `tools` 节点（即执行工具调用的节点）之前被中断。

In [ ]:
from langchain_deepseek import ChatDeepSeek

def multiply(a: int, b: int) -> int:
    """计算 a 和 b 的乘积。

    Args:
        a: 第一个整数
        b: 第二个整数
    """
    return a * b

# 这将作为一个工具
def add(a: int, b: int) -> int:
    """计算 a 和 b 的和。

    Args:
        a: 第一个整数
        b: 第二个整数
    """
    return a + b

def divide(a: int, b: int) -> float:
    """计算 a 除以 b 的商。

    Args:
        a: 第一个整数
        b: 第二个整数
    """
    return a / b

tools = [add, multiply, divide]
llm = ChatDeepSeek(model="deepseek-chat")
llm_with_tools = llm.bind_tools(tools)

In [ ]:
from IPython.display import Image, display

from langgraph.checkpoint.memory import MemorySaver
from langgraph.graph import MessagesState
from langgraph.graph import START, StateGraph
from langgraph.prebuilt import tools_condition, ToolNode

from langchain_core.messages import AIMessage, HumanMessage, SystemMessage

# 系统消息
sys_msg = SystemMessage(content="You are a helpful assistant tasked with performing arithmetic on a set of inputs.")

# 节点
def assistant(state: MessagesState):
   return {"messages": [llm_with_tools.invoke([sys_msg] + state["messages"])]}

# 图
builder = StateGraph(MessagesState)

# 定义节点：这些节点执行实际工作
builder.add_node("assistant", assistant)
builder.add_node("tools", ToolNode(tools))

# 定义边：这些边决定控制流
builder.add_edge(START, "assistant")
builder.add_conditional_edges(
    "assistant",
    # 如果来自 assistant 的最新消息（结果）是工具调用 -> tools_condition 路由到 tools
    # 如果来自 assistant 的最新消息（结果）不是工具调用 -> tools_condition 路由到 END
    tools_condition,
)
builder.add_edge("tools", "assistant")

memory = MemorySaver()
graph = builder.compile(interrupt_before=["tools"], checkpointer=memory)

# 展示
display(Image(graph.get_graph(xray=True).draw_mermaid_png()))

In [ ]:
# 输入
initial_input = {"messages": HumanMessage(content="Multiply 2 and 3")}

# 线程
thread = {"configurable": {"thread_id": "1"}}

# 运行图直到第一次中断
for event in graph.stream(initial_input, thread, stream_mode="values"):
    event['messages'][-1].pretty_print()

我们可以获取状态并查看下一个要调用的节点。

这是确认图已被中断的好方法。

In [ ]:
state = graph.get_state(thread)
state.next

现在，我们来介绍一个小技巧。

当我们以 `None` 作为输入调用图时，它将从上一次状态检查点继续执行！

![breakpoints.jpg](https://cdn.prod.website-files.com/65b8cd72835ceeacd4449a53/66dbae7985b747dfed67775d_breakpoints1.png)

为了清晰起见，LangGraph 会重新发出当前状态，其中包含带有工具调用的 `AIMessage`。

然后它会继续执行图中的后续步骤，从工具节点开始。

我们看到工具节点使用此工具调用运行，并将结果传回聊天模型以获取最终答案。

In [ ]:
for event in graph.stream(None, thread, stream_mode="values"):
    event['messages'][-1].pretty_print()

现在，让我们将这些整合到一个具体的用户审批步骤中，该步骤接受用户输入。

In [ ]:
# 输入
initial_input = {"messages": HumanMessage(content="Multiply 2 and 3")}

# 线程
thread = {"configurable": {"thread_id": "2"}}

# 运行图直到第一次中断
for event in graph.stream(initial_input, thread, stream_mode="values"):
    event['messages'][-1].pretty_print()

# 获取用户反馈
user_approval = input("是否要调用该工具？(yes/no): ")

# 检查审批
if user_approval.lower() == "yes":
    
    # 如果批准，继续图的执行
    for event in graph.stream(None, thread, stream_mode="values"):
        event['messages'][-1].pretty_print()
        
else:
    print("用户取消了操作。")

### 通过 LangGraph API 使用断点

**⚠️ 注意**

自录制这些视频以来，我们已更新了 Studio，现在可以在本地运行并通过浏览器访问。这是运行 Studio 的首选方式，而非视频中展示的桌面应用。它现在名为 _LangSmith Studio_（而非 _LangGraph Studio_）。详细设置说明请参见课程开头的"环境搭建"指南。你可以在[此处](https://docs.langchain.com/langsmith/studio)找到 Studio 的说明，在[此处](https://docs.langchain.com/langsmith/quick-start-studio#local-development-server)找到本地部署的具体细节。  
要启动本地开发服务器，请在本模块的 `/studio` 目录下的终端中运行以下命令：

```
langgraph dev
```

你应该看到以下输出：
```
- 🚀 API: http://127.0.0.1:2024
- 🎨 Studio UI: https://smith.langchain.com/studio/?baseUrl=http://127.0.0.1:2024
- 📚 API Docs: http://127.0.0.1:2024/docs
```

打开浏览器并导航到上面显示的 **Studio UI** URL。

LangGraph API [支持断点](https://docs.langchain.com/langsmith/add-human-in-the-loop)。

In [ ]:
if 'google.colab' in str(get_ipython()):
    raise Exception("Unfortunately LangGraph Studio is currently not supported on Google Colab")

In [ ]:
# 这是本地开发服务器的 URL
from langgraph_sdk import get_client
client = get_client(url="http://127.0.0.1:2024")

如上所示，我们可以在编译在 Studio 中运行的图时添加 `interrupt_before=["node"]`。

但是，通过 API，你也可以直接将 `interrupt_before` 传递给流式方法。

In [ ]:
initial_input = {"messages": HumanMessage(content="Multiply 2 and 3")}
thread = await client.threads.create()
async for chunk in client.runs.stream(
    thread["thread_id"],
    assistant_id="agent",
    input=initial_input,
    stream_mode="values",
    interrupt_before=["tools"],
):
    print(f"Receiving new event of type: {chunk.event}...")
    messages = chunk.data.get('messages', [])
    if messages:
        print(messages[-1])
    print("-" * 50)

现在，我们可以像之前一样从断点继续执行，只需传入 `thread_id` 和 `None` 作为输入！

In [ ]:
async for chunk in client.runs.stream(
    thread["thread_id"],
    "agent",
    input=None,
    stream_mode="values",
    interrupt_before=["tools"],
):
    print(f"Receiving new event of type: {chunk.event}...")
    messages = chunk.data.get('messages', [])
    if messages:
        print(messages[-1])
    print("-" * 50)